# Handwritten Character Recognition
CNN (PyTorch) trained on MNIST to classify handwritten digits (0-9). CodeAlpha Machine Learning Internship.

In [ ]:
import os
import json
import copy
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split
import torchvision

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
IMG_DIM = 28
NUM_CLASSES = 10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs('screenshots', exist_ok=True)
os.makedirs('models', exist_ok=True)


## Load Dataset (full MNIST via torchvision, falls back to load_digits offline, upscaled to 28x28)

In [ ]:
def _upscale_to_28(images_8x8):
    out = np.zeros((len(images_8x8), IMG_DIM, IMG_DIM), dtype='uint8')
    for i, img in enumerate(images_8x8):
        pil_img = Image.fromarray((img / 16.0 * 255).astype('uint8'))
        pil_img = pil_img.resize((IMG_DIM, IMG_DIM), Image.BICUBIC)
        out[i] = np.array(pil_img)
    return out

try:
    train_set = torchvision.datasets.MNIST(root='mnist_data', train=True, download=True)
    test_set = torchvision.datasets.MNIST(root='mnist_data', train=False, download=True)
    X_train = train_set.data.numpy()
    y_train = train_set.targets.numpy()
    X_test = test_set.data.numpy()
    y_test = test_set.targets.numpy()
    source = 'full MNIST (70,000 images, 28x28)'
except Exception as exc:
    print('Falling back to load_digits:', exc)
    from sklearn.datasets import load_digits
    from sklearn.model_selection import train_test_split
    digits = load_digits()
    X_all = _upscale_to_28(digits.images)
    y_all = digits.target.astype('int64')
    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y_all, test_size=0.2, random_state=RANDOM_SEED, stratify=y_all)
    source = 'sklearn load_digits (1,797 images, upscaled 8x8 -> 28x28)'

X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0
X_train = X_train.reshape(-1, 1, IMG_DIM, IMG_DIM)
X_test = X_test.reshape(-1, 1, IMG_DIM, IMG_DIM)
y_train = y_train.astype('int64')
y_test = y_test.astype('int64')
print('Dataset source:', source)
print('Train:', X_train.shape, 'Test:', X_test.shape)


## Visualize Sample Images

In [ ]:
plt.figure(figsize=(12, 3))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(X_train[i].reshape(IMG_DIM, IMG_DIM), cmap='gray')
    plt.title(str(y_train[i]))
    plt.axis('off')
plt.suptitle('Sample Handwritten Digit Training Images')
plt.tight_layout()
plt.savefig('screenshots/sample_images.png', dpi=150)
plt.show()


## Build CNN Model

In [ ]:
class HandwrittenCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2), nn.Dropout(0.25),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2), nn.Dropout(0.25),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * (IMG_DIM // 4) * (IMG_DIM // 4), 128), nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        # Returns raw logits (no softmax); use nn.CrossEntropyLoss for
        # training and torch.softmax(...) when you need probabilities.
        x = self.features(x)
        x = self.classifier(x)
        return x

model = HandwrittenCNN().to(DEVICE)
print(model)


## Train

In [ ]:
full_train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
val_size = int(0.1 * len(full_train_ds))
train_size = len(full_train_ds) - val_size
train_ds, val_ds = random_split(full_train_ds, [train_size, val_size],
                                 generator=torch.Generator().manual_seed(RANDOM_SEED))
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)

optimizer = torch.optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()

epochs, patience = 25, 5
history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}
best_val_loss, best_state, epochs_without_improvement = float('inf'), None, 0

for epoch in range(epochs):
    model.train()
    running_loss, running_correct, running_total = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
        running_correct += (outputs.argmax(1) == yb).sum().item()
        running_total += xb.size(0)
    train_loss, train_acc = running_loss / running_total, running_correct / running_total

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            outputs = model(xb)
            loss = criterion(outputs, yb)
            val_loss += loss.item() * xb.size(0)
            val_correct += (outputs.argmax(1) == yb).sum().item()
            val_total += xb.size(0)
    val_loss, val_acc = val_loss / val_total, val_correct / val_total

    history['loss'].append(train_loss); history['accuracy'].append(train_acc)
    history['val_loss'].append(val_loss); history['val_accuracy'].append(val_acc)
    print(f'Epoch {epoch+1}/{epochs} - loss: {train_loss:.4f} - accuracy: {train_acc:.4f} '
          f'- val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}')

    if val_loss < best_val_loss:
        best_val_loss, best_state, epochs_without_improvement = val_loss, copy.deepcopy(model.state_dict()), 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print(f'Early stopping at epoch {epoch+1} (best val_loss: {best_val_loss:.4f})')
            break

if best_state is not None:
    model.load_state_dict(best_state)


## Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(history['loss'], label='Training Loss', color='crimson')
axes[0].plot(history['val_loss'], label='Validation Loss', color='darkorange')
axes[0].set_title('Model Loss'); axes[0].legend()
axes[1].plot(history['accuracy'], label='Training Accuracy', color='seagreen')
axes[1].plot(history['val_accuracy'], label='Validation Accuracy', color='royalblue')
axes[1].set_title('Model Accuracy'); axes[1].legend()
plt.tight_layout()
plt.savefig('screenshots/training_history.png', dpi=150)
plt.show()


## Evaluate on Test Set

In [ ]:
model.eval()
test_loader = DataLoader(TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test)),
                          batch_size=256, shuffle=False)
all_preds = []
with torch.no_grad():
    for xb, _ in test_loader:
        xb = xb.to(DEVICE)
        outputs = model(xb)
        all_preds.append(outputs.argmax(1).cpu().numpy())
y_pred = np.concatenate(all_preds)

test_acc = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {test_acc:.4f}')
report = classification_report(y_test, y_pred, digits=4)
print(report)
with open('screenshots/classification_report.txt', 'w') as f:
    f.write(f'Dataset source: {source}\n')
    f.write(f'Test Accuracy: {test_acc:.4f}\n\n')
    f.write(report)


## Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=range(10), yticklabels=range(10))
plt.title('Confusion Matrix'); plt.xlabel('Predicted Label'); plt.ylabel('True Label')
plt.tight_layout()
plt.savefig('screenshots/confusion_matrix.png', dpi=150)
plt.show()


## Save Model

In [ ]:
torch.save(model.state_dict(), 'models/mnist_cnn.pt')
with open('models/meta.json', 'w') as f:
    json.dump({'img_dim': IMG_DIM, 'pixel_max': 255.0, 'dataset_source': source}, f)
print('Saved model and metadata.')
